# D2.2 · When the actor is an agent

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.1 · Agent-assisted reconstruction](https://spbreed.github.io/cyber-commons/lessons/D2.1.html)**.

| | |
|---|---|
| Tools used | Keycloak, OpenSearch |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

The internal actor was autonomous. Was it instructed, was it compromised, or did it simply do what it was allowed to do? None of your existing playbooks have a branch for that question, and the answer changes everything downstream.

> **At CyberTravels.** The internal actor was the Workflow Agent. Was it instructed, injected, or simply permitted? CyberTravels' existing playbook has no branch for that question, and every step of it assumes a person.

## 2 · The framework

```
   the internal actor was autonomous. which branch?

   instructed      someone told it to        -> who, and through what channel
   injected        content told it to        -> which corpus, written by whom
   permitted       it was allowed to         -> a control gap, not an intrusion

   no existing playbook has this branch, and it changes everything after it
```

Three responder instincts are correct for human incidents and misfire when the
actor is an agent.

1. **Disable the account.** For a human this stops them. For an agent holding an
   already-issued bearer token, it may not — the token remains valid until it
   expires.
2. **Interview the user.** They were asleep. They authorised a task; a model
   chose the actions. They cannot tell you what happened.
3. **Assume one actor.** There were three, in a chain, and only the last one
   touched the resource.

The correct first action is to **revoke the agent identity**, which is only
possible if A2 was done. This lesson is where the identity track's value becomes
operational rather than architectural.

## 3 · Demo — the three instincts, tested

In [ ]:
import time
from dataclasses import dataclass, field

@dataclass
class Session:
    actor: str; token_issued: float; token_ttl: float; account_enabled: bool = True
    identity_revoked: bool = False
    def can_act(self, at):
        if self.identity_revoked: return False, "identity revoked"
        if at - self.token_issued > self.token_ttl: return False, "token expired"
        if not self.account_enabled:
            return True, "account disabled, but the issued token is still valid"
        return True, "active"

now = time.time()
SESSIONS = {
 "dana@corp (human)":  Session("dana@corp", now-60, 3600),
 "patch-agent":        Session("patch-agent", now-60, 3600),
 "deploy-agent":       Session("deploy-agent", now-60, 3600),
}
print("INSTINCT 1 — disable dana@corp's account")
for s in SESSIONS.values(): s.account_enabled = (s.actor != "dana@corp")
for name, s in SESSIONS.items():
    ok, why = s.can_act(now)
    print(f"   {name:22s} can act: {str(ok):5s}  {why}")
print("   → the agents were never using her account interactively; they hold")
print("     their own issued tokens, and one of them is acting AS her.")

In [ ]:
print("\nINSTINCT 2 — interview the user")
INTERVIEW = {
 "did you read the AWS credentials?":       "No. I opened a ticket and went to lunch.",
 "what did you ask the agent to do?":       "Fix the finding in billing.py.",
 "did you approve the external POST?":      "I didn't know it made external calls.",
}
for q, a in INTERVIEW.items():
    print(f"   Q: {q}\n   A: {a}")
print("   → she authorised a TASK. The actions were chosen by a model. She is")
print("     not withholding information; she does not have it.")

print("\nINSTINCT 3 — assume one actor")
CHAIN = ["dana@corp", "orchestrator", "patch-agent"]
print(f"   actual chain: {' → '.join(CHAIN)}")
print(f"   actors involved: {len(CHAIN)}; actors in the logs: 1")

## 4 · The control — revoke the agent identity first

In [ ]:
class Registry:
    def __init__(self):
        self.revoked = set()
    def revoke(self, actor):
        self.revoked.add(actor); return actor
    def valid(self, session):
        return session.actor not in self.revoked

reg = Registry()
for s in SESSIONS.values(): s.account_enabled = True   # undo instinct 1

print("correct first action — revoke patch-agent's identity:")
reg.revoke("patch-agent")
SESSIONS["patch-agent"].identity_revoked = True
for name, s in SESSIONS.items():
    ok, why = s.can_act(now)
    print(f"   {name:22s} can act: {str(ok):5s}  {why}")
print("\n   dana keeps working. deploy-agent keeps working. The actor stopped.")

print("\nTIME TO EFFECT, measured:")
LEVERS = {"disable the human's account": (5,  "agent unaffected"),
          "kill the agent process":      (2,  "supervisor restarts it; token still valid"),
          "revoke the agent identity":   (12, "agent cannot act, even after restart"),
          "rotate the shared credential":(420,"works, and breaks every other consumer")}
for lever, (secs, note) in LEVERS.items():
    print(f"   {lever:32s}{secs:>5}s  {note}")
assert not SESSIONS["patch-agent"].can_act(now)[0]
assert SESSIONS["deploy-agent"].can_act(now)[0]

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">the runbook you have</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">the runbook this incident needs</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">1. disable the user account</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">1. identify the &lt;b&gt;acting&lt;/b&gt; identity from the act chain (A2.5)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">2. interview the user</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">2. revoke that identity — no approval needed for a non-human (A3.6)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">3. review the user&#x27;s recent activity</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">3. scope by walking the delegation chain, not the host list (D2.3)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">4. preserve the run trace before anything restarts (D2.5)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">5. only then consider the human&#x27;s account, and say why</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Every step on the left is correct for a human actor and wrong here. The human authorised a task; the actions were chosen by a model.</div>

## What you just proved

Disabling the human's account leaves both agents able to act on already-issued tokens. The interview establishes the user authorised a task, not the actions. The chain shows three actors where the logs show one. Revoking `patch-agent`'s identity stops it in 12 seconds while dana and `deploy-agent` continue working.

## Your turn

Write your agentic incident runbook's first three steps. If step one is "disable the user account", rewrite it — and check whether you can currently revoke a single agent identity at all.

---

**Next → [D2.3 · Scoping an agentic incident](https://spbreed.github.io/cyber-commons/lessons/D2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*